# 03 — Pandas Avancé : groupby, merge, apply, pivot

Ce notebook est le **miroir exact de `02_sql_advanced.ipynb`** : mêmes données (mêmes valeurs, même seed), mêmes questions métier — mais résolues avec pandas plutôt que SQL. L'objectif : pouvoir répondre en entretien à *« tu ferais ça en SQL ou en pandas ? »* en connaissance de cause, pas par réflexe.

## Sommaire
1. Setup — recréation des mêmes données en DataFrames
2. `merge` — jointures
3. `groupby` — agrégations régionales
4. `rank()` — équivalent de ROW_NUMBER / RANK / DENSE_RANK
5. `transform()` — équivalent de PARTITION BY et des subqueries corrélées
6. `shift()` — équivalent de LAG / LEAD
7. `rolling()` — moyenne mobile
8. `apply()` — logique métier personnalisée
9. `pivot_table()` — reformatage large
10. Pipeline chaîné — équivalent des CTEs chaînées
11. Récapitulatif comparatif SQL ↔ pandas


## 1. Setup — recréation des données

Mêmes données exactement que dans `02_sql_advanced.ipynb` (même seed `42`), cette fois directement en DataFrames pandas plutôt qu'en base SQLite.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# --- communes ---
communes = pd.DataFrame([
    (1, "Le Havre",   "Seine-Maritime", 170000, 0.9041, 21.4),
    (2, "Rouen",      "Seine-Maritime", 110000, 0.5451, 15.2),
    (3, "Dieppe",     "Seine-Maritime",  30000, 0.5500, 16.7),
    (4, "Caen",       "Calvados",       105000, 0.1771,  9.8),
    (5, "Evreux",     "Eure",            48000, 0.4200, 13.1),
    (6, "Cherbourg",  "Manche",          78000, 0.3100, 11.0),
], columns=["commune_id", "nom", "region", "population", "score_je", "taux_pauvrete"])

communes


,commune_id,nom,region,population,score_je,taux_pauvrete
0,1,Le Havre,Seine-Maritime,170000,0.9041,21.4
1,2,Rouen,Seine-Maritime,110000,0.5451,15.2
2,3,Dieppe,Seine-Maritime,30000,0.5500,16.7
3,4,Caen,Calvados,105000,0.1771,9.8
4,5,Evreux,Eure,48000,0.4200,13.1
5,6,Cherbourg,Manche,78000,0.3100,11.0


In [2]:
# --- releves_pollution : 12 mois de relevés NO2 par commune ---
base_no2 = {1: 38.2, 2: 27.5, 3: 25.9, 4: 14.1, 5: 22.0, 6: 18.4}
mois_liste = [f"2025-{m:02d}" for m in range(1, 13)]

rows = []
for commune_id, base in base_no2.items():
    for i, mois in enumerate(mois_liste):
        saison = 6 * np.cos(2 * np.pi * i / 12)
        bruit = np.random.normal(0, 1.5)
        valeur = round(base + saison + bruit, 1)
        rows.append((commune_id, mois, valeur))

releves = pd.DataFrame(rows, columns=["commune_id", "mois", "no2"])
releves.head(10)


,commune_id,mois,no2
0,1,2025-01,44.9
1,1,2025-02,43.2
2,1,2025-03,42.2
3,1,2025-04,40.5
4,1,2025-05,34.8
5,1,2025-06,32.7
6,1,2025-07,34.6
7,1,2025-08,34.2
8,1,2025-09,34.5
9,1,2025-10,39.0


## 2. `merge` — l'équivalent pandas de JOIN

`pd.merge()` reproduit exactement la logique d'un `JOIN` SQL. Les paramètres `how="inner"`, `"left"`, `"right"`, `"outer"` correspondent directement à `INNER JOIN`, `LEFT JOIN`, etc.

In [3]:
df = releves.merge(communes, on="commune_id", how="inner")
df.head()


,commune_id,mois,no2,nom,region,population,score_je,taux_pauvrete
0,1,2025-01,44.9,Le Havre,Seine-Maritime,170000,0.9041,21.4
1,1,2025-02,43.2,Le Havre,Seine-Maritime,170000,0.9041,21.4
2,1,2025-03,42.2,Le Havre,Seine-Maritime,170000,0.9041,21.4
3,1,2025-04,40.5,Le Havre,Seine-Maritime,170000,0.9041,21.4
4,1,2025-05,34.8,Le Havre,Seine-Maritime,170000,0.9041,21.4


> 💡 Équivalent SQL : `SELECT * FROM releves_pollution r JOIN communes c ON c.commune_id = r.commune_id`

## 3. `groupby` — agrégations régionales

**Question métier (identique à 3.1 dans le notebook SQL) :** quelle est la moyenne de NO2 par région ?

In [4]:
moyenne_regionale = df.groupby("region")["no2"].mean().round(1)
moyenne_regionale


region
Calvados          13.6
Eure              22.1
Manche            18.6
Seine-Maritime    30.3
Name: no2, dtype: float64

> 💡 Équivalent SQL : `SELECT region, AVG(no2) FROM ... GROUP BY region`

## 4. `rank()` — équivalent de ROW_NUMBER / RANK / DENSE_RANK

Le paramètre `method` de `.rank()` contrôle exactement le même comportement que le choix entre `ROW_NUMBER()`, `RANK()` et `DENSE_RANK()` en SQL :
- `method="first"` → équivalent `ROW_NUMBER()` (jamais d'égalité)
- `method="min"` → équivalent `RANK()` (laisse des trous)
- `method="dense"` → équivalent `DENSE_RANK()` (pas de trous)

In [5]:
classement = communes[["nom", "score_je"]].copy()
classement["rang_row_number"] = classement["score_je"].rank(ascending=False, method="first").astype(int)
classement.sort_values("rang_row_number")


,nom,score_je,rang_row_number
0,Le Havre,0.9041,1
2,Dieppe,0.5500,2
1,Rouen,0.5451,3
4,Evreux,0.4200,4
5,Cherbourg,0.3100,5
3,Caen,0.1771,6


In [6]:
# Avec égalités artificielles (comme dans le notebook SQL, sur taux_pauvrete arrondi)
comparaison = communes[["nom", "taux_pauvrete"]].copy()
comparaison["taux_arrondi"] = comparaison["taux_pauvrete"].round()
comparaison["rang_avec_trous"] = comparaison["taux_arrondi"].rank(ascending=False, method="min").astype(int)
comparaison["rang_sans_trous"] = comparaison["taux_arrondi"].rank(ascending=False, method="dense").astype(int)
comparaison.sort_values("rang_avec_trous")


,nom,taux_pauvrete,taux_arrondi,rang_avec_trous,rang_sans_trous
0,Le Havre,21.4,21.0,1,1
2,Dieppe,16.7,17.0,2,2
1,Rouen,15.2,15.0,3,3
4,Evreux,13.1,13.0,4,4
5,Cherbourg,11.0,11.0,5,5
3,Caen,9.8,10.0,6,6


## 5. `transform()` — équivalent de PARTITION BY et des subqueries corrélées

C'est LA méthode pandas la plus proche conceptuellement d'une window function SQL avec `PARTITION BY` : elle calcule une valeur par groupe, **mais renvoie un résultat de la même taille que le DataFrame d'origine** (contrairement à `groupby().agg()` qui réduit les lignes) — exactement le comportement d'une window function.

**Question métier (identique à 3.1 / 4.1 du notebook SQL) :** quelles communes dépassent la moyenne de NO2 (ou de score JE) de leur propre région ?

In [7]:
communes["score_je_moyen_region"] = communes.groupby("region")["score_je"].transform("mean")
communes["au_dessus_moyenne_region"] = communes["score_je"] > communes["score_je_moyen_region"]
communes[["nom", "region", "score_je", "score_je_moyen_region", "au_dessus_moyenne_region"]]


,nom,region,score_je,score_je_moyen_region,au_dessus_moyenne_region
0,Le Havre,Seine-Maritime,0.9041,0.6664,True
1,Rouen,Seine-Maritime,0.5451,0.6664,False
2,Dieppe,Seine-Maritime,0.5500,0.6664,False
3,Caen,Calvados,0.1771,0.1771,False
4,Evreux,Eure,0.4200,0.4200,False
5,Cherbourg,Manche,0.3100,0.3100,False


> 💡 C'est exactement l'équivalent pandas de la **subquery corrélée** du notebook SQL : `transform` recalcule la moyenne pour chaque ligne en fonction de son propre groupe — même logique, syntaxe bien plus concise.

**PARTITION BY pour un pic (le relevé NO2 le plus élevé par commune)** — équivalent pandas :

In [8]:
idx_max = df.groupby("commune_id")["no2"].idxmax()
pics = df.loc[idx_max, ["nom", "mois", "no2"]].sort_values("no2", ascending=False)
pics


,nom,mois,no2
0,Le Havre,2025-01,44.9
12,Rouen,2025-01,33.9
25,Dieppe,2025-02,31.3
59,Evreux,2025-12,28.7
71,Cherbourg,2025-12,25.9
47,Caen,2025-12,20.9


## 6. `shift()` — équivalent de LAG / LEAD

`shift(1)` recule les valeurs d'une ligne (équivalent `LAG`), `shift(-1)` les avance d'une ligne (équivalent `LEAD`). **Important** : bien trier avec `sort_values` et grouper avec `groupby` avant, sinon le décalage se fait sur l'ordre brut du DataFrame plutôt que sur la vraie chronologie par commune.

In [9]:
le_havre = df[df["nom"] == "Le Havre"].sort_values("mois").copy()
le_havre["no2_mois_precedent"] = le_havre["no2"].shift(1)
le_havre["variation"] = (le_havre["no2"] - le_havre["no2_mois_precedent"]).round(1)
le_havre[["mois", "no2", "no2_mois_precedent", "variation"]]


,mois,no2,no2_mois_precedent,variation
0,2025-01,44.9,NaN,NaN
1,2025-02,43.2,44.9,-1.7
2,2025-03,42.2,43.2,-1.0
3,2025-04,40.5,42.2,-1.7
4,2025-05,34.8,40.5,-5.7
5,2025-06,32.7,34.8,-2.1
6,2025-07,34.6,32.7,1.9
7,2025-08,34.2,34.6,-0.4
8,2025-09,34.5,34.2,0.3
9,2025-10,39.0,34.5,4.5


**Sur toutes les communes à la fois**, il faut grouper avant de décaler pour ne pas mélanger la fin d'une commune avec le début d'une autre :

In [10]:
df_sorted = df.sort_values(["commune_id", "mois"]).copy()
df_sorted["no2_mois_precedent"] = df_sorted.groupby("commune_id")["no2"].shift(1)
df_sorted[df_sorted["nom"] == "Rouen"][["nom", "mois", "no2", "no2_mois_precedent"]].head(6)


,nom,mois,no2,no2_mois_precedent
12,Rouen,2025-01,33.9,NaN
13,Rouen,2025-02,29.8,33.9
14,Rouen,2025-03,27.9,29.8
15,Rouen,2025-04,26.7,27.9
16,Rouen,2025-05,23.0,26.7
17,Rouen,2025-06,22.8,23.0


## 7. `rolling()` — moyenne mobile

Équivalent direct de `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` : `.rolling(window=3)` calcule sur la ligne courante + les 2 précédentes.

In [11]:
le_havre["moyenne_mobile_3mois"] = le_havre["no2"].rolling(window=3).mean().round(1)
le_havre[["mois", "no2", "moyenne_mobile_3mois"]]


,mois,no2,moyenne_mobile_3mois
0,2025-01,44.9,NaN
1,2025-02,43.2,NaN
2,2025-03,42.2,43.4
3,2025-04,40.5,42.0
4,2025-05,34.8,39.2
5,2025-06,32.7,36.0
6,2025-07,34.6,34.0
7,2025-08,34.2,33.8
8,2025-09,34.5,34.4
9,2025-10,39.0,35.9


## 8. `apply()` — logique métier personnalisée

`apply()` exécute une fonction Python arbitraire ligne par ligne (ou groupe par groupe) — utile quand la transformation est trop spécifique pour une fonction pandas native. À utiliser avec parcimonie : plus lent que les opérations vectorisées natives (`transform`, `rank`, etc.), donc réservé aux cas où il n'y a pas d'alternative vectorisée simple.

**Exemple :** reclassifier chaque commune en catégorie de sévérité selon son score JE (comme dans `src/models/clustering.py` du projet EcoSense).

In [12]:
def categoriser_severite(score):
    if score >= 0.7:
        return "CRITIQUE"
    elif score >= 0.35:
        return "MODÉRÉ"
    else:
        return "BON"

communes["severite"] = communes["score_je"].apply(categoriser_severite)
communes[["nom", "score_je", "severite"]].sort_values("score_je", ascending=False)


,nom,score_je,severite
0,Le Havre,0.9041,CRITIQUE
2,Dieppe,0.5500,MODÉRÉ
1,Rouen,0.5451,MODÉRÉ
4,Evreux,0.4200,MODÉRÉ
5,Cherbourg,0.3100,BON
3,Caen,0.1771,BON


> 💡 Équivalent SQL : un `CASE WHEN ... THEN ... ELSE ... END` — mais `apply()` permet une logique Python arbitrairement complexe (boucles, appels à d'autres fonctions), pas seulement des comparaisons simples.

## 9. `pivot_table()` — reformatage large (wide format)

Transforme un format "long" (une ligne par mesure) en format "large" (une colonne par catégorie) — très utile pour comparer visuellement plusieurs communes sur la même période, ou avant d'exporter vers Excel.

In [13]:
pivot = df.pivot_table(index="mois", columns="nom", values="no2")
pivot.round(1)


nom,Caen,Cherbourg,Dieppe,Evreux,Le Havre,Rouen
mois,,,,,,
2025-01,20.4,23.7,31.1,28.5,44.9,33.9
2025-02,16.4,23.3,31.3,24.6,43.2,29.8
2025-03,15.1,19.7,27.2,25.5,42.2,27.9
2025-04,14.4,16.6,26.5,21.4,40.5,26.7
2025-05,12.2,16.6,22.0,18.0,34.8,23.0
2025-06,9.2,15.2,20.3,17.7,32.7,22.8
2025-07,7.9,12.3,19.0,17.5,34.6,20.1
2025-08,8.5,14.7,23.5,18.2,34.2,20.2
2025-09,8.9,15.9,22.9,17.7,34.5,26.7


**Bonus** : avec un `aggfunc`, `pivot_table` peut aussi agréger (comme un `GROUP BY` + pivot combinés) :

In [14]:
pivot_agg = df.pivot_table(index="region", columns="nom", values="no2", aggfunc="mean")
pivot_agg.round(1)


nom,Caen,Cherbourg,Dieppe,Evreux,Le Havre,Rouen
region,,,,,,
Calvados,13.6,NaN,NaN,NaN,NaN,NaN
Eure,NaN,NaN,NaN,22.1,NaN,NaN
Manche,NaN,18.6,NaN,NaN,NaN,NaN
Seine-Maritime,NaN,NaN,25.6,NaN,38.6,26.6


## 10. Pipeline chaîné — équivalent des CTEs chaînées

**Question métier (identique à 3.2 du notebook SQL) :** quelles communes cumulent un score JE au-dessus de la médiane ET un NO2 moyen annuel supérieur à 20 µg/m³ ?

En pandas, on enchaîne les étapes avec des variables intermédiaires nommées — le même principe de lisibilité qu'une CTE, juste avec une syntaxe différente.

In [15]:
# Étape 1 : NO2 moyen annuel par commune (équivalent de la 1ère CTE)
no2_annuel = releves.groupby("commune_id")["no2"].mean().rename("no2_moyen_annuel")

# Étape 2 : enrichissement (équivalent de la 2e CTE)
mediane_score_je = communes["score_je"].median()
communes_enrichies = communes.merge(no2_annuel, on="commune_id")
communes_enrichies["position_score_je"] = np.where(
    communes_enrichies["score_je"] >= mediane_score_je,
    "Au-dessus médiane", "En-dessous médiane"
)

# Étape 3 : filtre final
resultat = communes_enrichies[
    (communes_enrichies["position_score_je"] == "Au-dessus médiane") &
    (communes_enrichies["no2_moyen_annuel"] > 20)
].sort_values("no2_moyen_annuel", ascending=False)

resultat[["nom", "score_je", "no2_moyen_annuel", "position_score_je"]]


,nom,score_je,no2_moyen_annuel,position_score_je
0,Le Havre,0.9041,38.650,Au-dessus médiane
1,Rouen,0.5451,26.625,Au-dessus médiane
2,Dieppe,0.5500,25.625,Au-dessus médiane


## 11. Récapitulatif comparatif SQL ↔ pandas

| Besoin | SQL | pandas |
|---|---|---|
| Jointure | `JOIN` | `pd.merge()` |
| Agrégation par groupe | `GROUP BY` + `AVG/COUNT/SUM` | `groupby().agg()` |
| Classement (sans égalité) | `ROW_NUMBER()` | `.rank(method="first")` |
| Classement (avec trous) | `RANK()` | `.rank(method="min")` |
| Classement (sans trous) | `DENSE_RANK()` | `.rank(method="dense")` |
| Calcul par groupe, sans réduire les lignes | `... OVER (PARTITION BY ...)` | `.transform()` |
| Valeur ligne précédente/suivante | `LAG()` / `LEAD()` | `.shift(1)` / `.shift(-1)` |
| Moyenne mobile | `ROWS BETWEEN n PRECEDING AND CURRENT ROW` | `.rolling(window=n).mean()` |
| Logique conditionnelle simple | `CASE WHEN ... END` | `np.where()` ou `.apply()` |
| Reformatage long → large | *(peu naturel en SQL pur)* | `.pivot_table()` |
| Étapes nommées chaînées | CTEs (`WITH ... , ... AS (...)`) | Variables intermédiaires nommées |

## Quand choisir l'un plutôt que l'autre ?

- **SQL** : quand les données vivent déjà en base (pas de coût de chargement), pour des agrégations lourdes déléguées au moteur de la base (souvent plus optimisé que pandas sur de gros volumes), ou quand plusieurs personnes/outils doivent interroger la même source.
- **pandas** : quand les données sont déjà en mémoire, pour des transformations complexes avec de la logique Python (ML, fonctions personnalisées), ou pour l'exploration itérative rapide en notebook.

En entretien, la bonne réponse n'est presque jamais "je préfère X" mais *"ça dépend d'où vivent les données et de ce qu'on veut faire ensuite"* — sais le justifier au cas par cas.

## Prochaines étapes

- ✅ `01_sql_basics.ipynb` — terminé
- ✅ `02_sql_advanced.ipynb` — terminé
- ✅ `03_pandas_advanced.ipynb` — terminé (ce notebook)
- 📅 Volet Oracle (`04_oracle_specifics.ipynb`) — recréer une sélection de requêtes du notebook 02 sur une vraie base Oracle via DataGrip
